# BRFSS 2025 — Data Preprocessing

### Objective

The objective of this notebook is to process the 2025
Behavioral Risk Factor Surveillance System (BRFSS) dataset
and calculate state-level obesity and diabetes prevalence.

The processed dataset will be used as input features (X)
in our Multiple Linear Regression model to predict
Medicaid GLP-1 prescription rates.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import altair as alt

## 2. Load the Dataset

In [2]:
df = pd.read_sas("LLCP2025.XPT_", format="xport")
df.head(5)

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,_VEGRES1,_FRUTSU1,_VEGESU1,_FRTLT1A,_VEGLT1A,_FRT16A,_VEG23A,_FRUITE1,_VEGETE1,_AIDTST4
0,1.0,1.0,b'04072025',b'04',b'07',b'2025',1100.0,b'2025000001',2.025000e+09,1.0,...,1.0,100.0,130.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79,1.0
1,1.0,1.0,b'04072025',b'04',b'07',b'2025',1100.0,b'2025000002',2.025000e+09,1.0,...,1.0,107.0,145.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79,2.0
2,1.0,1.0,b'04042025',b'04',b'04',b'2025',1100.0,b'2025000003',2.025000e+09,1.0,...,1.0,72.0,144.0,2.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79,2.0
3,1.0,1.0,b'04082025',b'04',b'08',b'2025',1100.0,b'2025000004',2.025000e+09,1.0,...,1.0,300.0,63.0,1.0,2.0,1.0,1.0,5.397605e-79,5.397605e-79,2.0
4,1.0,1.0,b'04082025',b'04',b'08',b'2025',1100.0,b'2025000005',2.025000e+09,1.0,...,1.0,100.0,700.0,1.0,1.0,1.0,1.0,5.397605e-79,5.397605e-79,2.0


In [3]:
# Check the dataset dimensions
print(df.shape)

(356158, 283)


## 3. Inspect Dataset Structure

In [4]:
# Check column names
print(df.columns.tolist())

# Check data types and non-null counts
df.info()

['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENM1', 'PVTRESD1', 'COLGHOUS', 'STATERE1', 'CELPHON1', 'LADULT1', 'NUMADULT', 'RESPSLC1', 'LANDSEX4', 'SAFETIME', 'CTELNUM1', 'CELLFON5', 'CADULT1', 'CELLSEX4', 'PVTRESD3', 'CCLGHOUS', 'CSTATE1', 'LANDLINE', 'HHADULT', 'SEXVAR', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'PRIMINS2', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'EXERANY2', 'BPHIGH6', 'BPMEDS1', 'CHOLCHK3', 'TOLDHI3', 'CHOLMED3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNC1', 'CHCOCNC1', 'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'HAVARTH4', 'DIABETE4', 'DIABAGE4', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL4', 'NUMPHON4', 'CPDEMO1C', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME3', 'PREGNANT', 'WEIGHT2', 'HEIGHT3', 'DEAF', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES', 'DIFFALON', 'SLEPTIM1', 'SMOKE100', 'SMOKDAY2', 'USENOW3', 'ECIGNOW3', 'ALCDAY4', 'AVEDRNK4', 'DRNK3GE5', 'MAXDRNKS', 'FLUSHOT7', 'FLSHTMY3', 'PNEUVAC4', 'TETANUS1

### 3.1 Variable Selection

Select the variables relevant to our research objectives and create a subset of the original BRFSS dataset.

In [5]:
# Select relevant variables
selected_columns = [
    "_STATE",
    "_BMI5CAT",
    "DIABETE4",
    "PRIMINS2",
    "_LLCPWT",
    "_STSTR",
    "_PSU"
]

brfss = df[selected_columns].copy()

In [6]:
# Check the dimensions of the selected dataset
print(brfss.shape)

(356158, 7)


In [7]:
# Check dataset information
brfss.info()

<class 'pandas.DataFrame'>
RangeIndex: 356158 entries, 0 to 356157
Data columns (total 7 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _STATE    356158 non-null  float64
 1   _BMI5CAT  323596 non-null  float64
 2   DIABETE4  356152 non-null  float64
 3   PRIMINS2  356156 non-null  float64
 4   _LLCPWT   356158 non-null  float64
 5   _STSTR    356158 non-null  float64
 6   _PSU      356158 non-null  float64
dtypes: float64(7)
memory usage: 19.0 MB


- `_BMI5CAT` has 32,562 missing values

In [8]:
brfss.head()

,_STATE,_BMI5CAT,DIABETE4,PRIMINS2,_LLCPWT,_STSTR,_PSU
0,1.0,4.0,3.0,3.0,135.411848,11011.0,2.025000e+09
1,1.0,2.0,1.0,7.0,190.376819,11011.0,2.025000e+09
2,1.0,4.0,1.0,3.0,471.630521,11011.0,2.025000e+09
3,1.0,3.0,1.0,3.0,473.720063,11011.0,2.025000e+09
4,1.0,3.0,3.0,1.0,662.064923,11011.0,2.025000e+09


### 3.2 Missing Value Percentages

In [9]:
# Missing values summary
missing_summary = pd.DataFrame({
    "Missing Count": brfss.isna().sum(),
    "Missing Percentage (%)": brfss.isna().mean() * 100
})

display(missing_summary.round(4))

,Missing Count,Missing Percentage (%)
_STATE,0,0.0000
_BMI5CAT,32562,9.1426
DIABETE4,6,0.0017
PRIMINS2,2,0.0006
_LLCPWT,0,0.0000
_STSTR,0,0.0000
_PSU,0,0.0000


### 3.3 Response Code Distribution
Examine the distribution of response codes to identify valid responses and special response categories before data cleaning and recoding.

#### 3.3.1 `_BMI5CAT`

In [10]:
# Check the distribution of BMI categories
brfss["_BMI5CAT"].value_counts(dropna=False).sort_index()

_BMI5CAT
1.0      5552
2.0     96272
3.0    114838
4.0    106934
NaN     32562
Name: count, dtype: int64

**Interpretation**

- `Codes 1–3` represent non-obese respondents.
- `Code 4` represents **obese respondents (BMI ≥ 30).**
- A total of 323,596 respondents have valid BMI categories.
- Missing BMI values will be excluded from obesity prevalence estimation.
- BMI categories will be recoded into a binary variable:
  - 1 = Obese
  - 0 = Non-obese

#### 3.3.2 `DIABETE4`

In [11]:
# Check the distribution of diabetes response codes
brfss["DIABETE4"].value_counts(dropna=False).sort_index()

DIABETE4
1.0     51827
2.0      2671
3.0    290712
4.0     10166
7.0       580
9.0       196
NaN         6
Name: count, dtype: int64

**Interpretation**

- `Code 1` represents respondents **diagnosed with diabetes.**
- `Codes 2, 3, and 4` are classified as non-diabetic
  for the purpose of this study.
- `Codes 7 and 9` represent uncertain or refused responses.
- A total of 355,376 respondents have valid responses.
- Invalid and missing responses will be excluded from
  diabetes prevalence estimation.
- Diabetes status will be recoded into a binary variable:
  - 1 = Diagnosed diabetes
  - 0 = No diagnosed diabetes, excluding pregnancy-only diabetes

#### 3.3.3 `PRIMINS2`

In [12]:
# Check the distribution of primary health insurance
brfss["PRIMINS2"].value_counts(dropna=False).sort_index()

PRIMINS2
1.0     116895
2.0      30073
3.0     117118
4.0        290
5.0      26732
6.0        118
7.0      14087
8.0       1049
9.0       8524
10.0      6852
77.0     10548
88.0     18958
99.0      4912
NaN          2
Name: count, dtype: int64

**Interpretation**

- `Code 5` identifies respondents who report **Medicaid
  as their primary health insurance.**
- A total of 26,732 respondents report Medicaid as
  their primary health insurance, representing
  approximately **7.51%** of the original sample.
- The Medicaid subpopulation will be further examined
  to assess state-level sample sizes and the feasibility
  of estimating Medicaid-specific prevalence.

### 3.4 Geographic Coverage

Examine the geographic coverage of the BRFSS dataset
to determine which states and territories are represented.

In [13]:
# Check the number of geographic areas
brfss["_STATE"].nunique()

50

In [14]:
# Count respondents by state identifier
brfss["_STATE"].value_counts().sort_index()

_STATE
1.0      4171
2.0      5537
4.0      5374
5.0      5119
8.0      9437
9.0      7444
10.0     4202
11.0     2622
12.0    13458
13.0     6191
15.0     7371
16.0     3588
17.0    11219
18.0     6550
19.0     8051
20.0     9912
21.0     7459
22.0     4899
23.0    12206
24.0    12943
25.0     9497
26.0    10173
27.0    14632
29.0     5939
30.0     5853
31.0     6968
33.0     5997
34.0     6739
35.0     3413
36.0    14876
37.0     3529
38.0     5729
39.0     9000
40.0     4196
41.0     4732
42.0     3017
44.0     5435
45.0     4137
46.0     4135
47.0     5731
48.0     6250
49.0    11232
50.0     6468
51.0     5017
53.0    19499
54.0     4215
55.0    11687
56.0     4537
66.0     1928
72.0     3844
Name: count, dtype: int64

#### 3.4.1 State Code Mapping

The `_STATE` variable contains numeric **State FIPS codes**.

To improve readability and facilitate integration with
other datasets, these numeric codes are mapped to their
corresponding state names and two-letter abbreviations.

In [15]:
# Define State FIPS code mapping based on the 2025 BRFSS Codebook
state_mapping = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    56: "Wyoming",
    66: "Guam",
    72: "Puerto Rico"
}

In [16]:
# Map numeric state codes to state names
brfss["State_Name"] = brfss["_STATE"].map(state_mapping)

In [17]:
# Verify the state mapping
display(brfss[["_STATE", "State_Name"]].head())

,_STATE,State_Name
0,1.0,Alabama
1,1.0,Alabama
2,1.0,Alabama
3,1.0,Alabama
4,1.0,Alabama


In [18]:
# Check for unmapped state codes
brfss.loc[brfss["State_Name"].isna(), "_STATE"].unique()

array([], dtype=float64)

#### 3.4.2 Overall Sample Size by State

In [19]:
# Count respondents by state
state_counts = brfss["State_Name"].value_counts().reset_index()

state_counts.columns = ["State", "Count"]

display(state_counts)

,State,Count
0,Washington,19499
1,New York,14876
2,Minnesota,14632
3,Florida,13458
4,Maryland,12943
5,Maine,12206
6,Wisconsin,11687
7,Utah,11232
8,Illinois,11219
9,Michigan,10173


In [20]:
alt.Chart(state_counts).mark_bar().encode(
    x=alt.X("Count:Q", title="Number of Respondents"),
    y=alt.Y("State:N", sort="-x", title="State or Territory"),
    tooltip=["State:N", "Count:Q"]
).properties(
    title="BRFSS 2025: Sample Size by State or Territory",
    width=600,
    height=800
)

alt.Chart(...)

#### 3.4.3 Medicaid Sample Size by State

Examine the number of respondents who reported Medicaid
as their primary health insurance coverage (`PRIMINS2` = 5)
in each state or territory.

This assessment provides an initial understanding of
whether sufficient sample sizes are available for
Medicaid-specific prevalence estimation.

In [21]:
# Count Medicaid respondents by state
medicaid_counts = (
    brfss.loc[brfss["PRIMINS2"] == 5, "State_Name"]
    .value_counts()
    .reindex(state_counts["State"], fill_value=0)
    .reset_index()
)

medicaid_counts.columns = ["State", "Count"]

display(medicaid_counts)

,State,Count
0,Washington,882
1,New York,2433
2,Minnesota,425
3,Florida,608
4,Maryland,745
5,Maine,1195
6,Wisconsin,415
7,Utah,511
8,Illinois,804
9,Michigan,905


##### 3.4.3.1 Visualization

In [22]:
alt.Chart(medicaid_counts).mark_bar().encode(
    x=alt.X("Count:Q", title="Number of Medicaid Respondents"),
    y=alt.Y("State:N", sort="-x", title="State or Territory"),
    tooltip=["State:N", "Count:Q"]
).properties(
    title="BRFSS 2025: Medicaid Sample Size by State",
    width=600,
    height=800
)

alt.Chart(...)

#### 3.4.4 Merge Sample Size Data

Combine the overall and Medicaid sample sizes by state
to compare the geographic distribution of both groups.

In [23]:
# Merge overall and Medicaid sample sizes
sample_comparison = state_counts.merge(
    medicaid_counts,
    on="State",
    suffixes=("_Total", "_Medicaid")
)

display(sample_comparison)

,State,Count_Total,Count_Medicaid
0,Washington,19499,882
1,New York,14876,2433
2,Minnesota,14632,425
3,Florida,13458,608
4,Maryland,12943,745
5,Maine,12206,1195
6,Wisconsin,11687,415
7,Utah,11232,511
8,Illinois,11219,804
9,Michigan,10173,905


In [24]:
# Calculate Medicaid sample proportion
sample_comparison["Medicaid_Pct"] = (
    sample_comparison["Count_Medicaid"]
    / sample_comparison["Count_Total"]
    * 100
)

display(sample_comparison.round(2))

,State,Count_Total,Count_Medicaid,Medicaid_Pct
0,Washington,19499,882,4.52
1,New York,14876,2433,16.36
2,Minnesota,14632,425,2.90
3,Florida,13458,608,4.52
4,Maryland,12943,745,5.76
5,Maine,12206,1195,9.79
6,Wisconsin,11687,415,3.55
7,Utah,11232,511,4.55
8,Illinois,11219,804,7.17
9,Michigan,10173,905,8.90


##### 3.4.4.1 Visualization

In [25]:
alt.Chart(sample_comparison).mark_bar().encode(
    x=alt.X(
        "Medicaid_Pct:Q",
        title="Medicaid Sample Proportion (%)"
    ),
    y=alt.Y(
        "State:N",
        sort="-x",
        title="State or Territory"
    ),
    tooltip=[
        "State:N",
        "Count_Total:Q",
        "Count_Medicaid:Q",
        alt.Tooltip("Medicaid_Pct:Q", format=".2f")
    ]
).properties(
    title="BRFSS 2025: Medicaid Sample Proportion by State",
    width=600,
    height=800
)

alt.Chart(...)

#### 3.4.5 Interpretation

**All 50 geographic areas contain respondents who reported
Medicaid as their primary health insurance coverage.**

However, Medicaid sample sizes vary considerably across
geographic areas, ranging from 116 to 2,433 respondents.

The unweighted Medicaid sample proportions also vary,
ranging from 2.56% in Wyoming to 35.09% in Puerto Rico.

These differences highlight the need to assess the
reliability of Medicaid-specific prevalence estimates
before selecting the final analytical population.

### 3.5 Survey Weight Quality Assessment

Examine survey weights and sampling design variables
to ensure that the data are suitable for weighted
state-level prevalence estimation.

#### 3.5.1 Descriptive Statistics of Survey Weights

Examine the distribution and descriptive statistics
of the final survey weight (`_LLCPWT`).

In [26]:
# Examine descriptive statistics of survey weights
brfss["_LLCPWT"].describe()

count    356158.000000
mean        667.437160
std        1221.855859
min           0.024484
25%         132.193760
50%         318.616880
75%         719.623302
max       45875.127317
Name: _LLCPWT, dtype: float64

#### 3.5.2 Invalid Survey Weight Assessment

Check for missing, zero, negative, and infinite
survey weights before weighted prevalence estimation.

In [27]:
# Check for invalid survey weights
weight = brfss["_LLCPWT"]

pd.Series({
    "Missing Weights": weight.isna().sum(),
    "Zero Weights": (weight == 0).sum(),
    "Negative Weights": (weight < 0).sum(),
    "Infinite Weights": np.isinf(weight).sum()
})

Missing Weights     0
Zero Weights        0
Negative Weights    0
Infinite Weights    0
dtype: int64

#### 3.5.3 Visualization

In [28]:
# Create histogram bins for valid survey weights
valid_weights = weight[np.isfinite(weight) & (weight > 0)]

counts, bins = np.histogram(valid_weights, bins=50)

weight_distribution = pd.DataFrame({
    "Bin_Start": bins[:-1],
    "Bin_End": bins[1:],
    "Count": counts
})

In [29]:
alt.Chart(weight_distribution).mark_bar().encode(
    x=alt.X("Bin_Start:Q", bin="binned", title="Survey Weight"),
    x2="Bin_End:Q",
    y=alt.Y("Count:Q", title="Number of Respondents"),
    tooltip=[
        alt.Tooltip("Bin_Start:Q", format=".2f"),
        alt.Tooltip("Bin_End:Q", format=".2f"),
        "Count:Q"
    ]
).properties(
    title="BRFSS 2025: Distribution of Survey Weights",
    width=600,
    height=350
)

alt.Chart(...)

#### 3.5.4 Interpretation of Survey Weights

All 356,158 respondents have valid, positive, and finite
survey weights, with **no missing or invalid values**.

The mean survey weight (667.44) is considerably higher
than the median (318.62), suggesting a right-skewed
distribution.

Although survey weights vary substantially across
respondents, this variability does not necessarily
indicate data quality issues.

Therefore, all original survey weights are retained
for weighted prevalence estimation.

---

#### 3.5.5 Sampling Design Variable Assessment

Examine sampling design variables to assess their
completeness and structure before evaluating the
reliability of weighted prevalence estimates.

##### 3.5.5.1 Missing Values and Unique Identifiers

Check for missing values and examine the number of
unique sampling strata and primary sampling units.

In [30]:
# Examine sampling design variables
design_columns = ["_STSTR", "_PSU"]

design_summary = pd.DataFrame({
    "Missing_Count": brfss[design_columns].isna().sum(),
    "Unique_Count": brfss[design_columns].nunique()
})

display(design_summary)

,Missing_Count,Unique_Count
_STSTR,0,2090
_PSU,0,19779


##### 3.5.5.2 Sampling Design Structure by State

Examine the number of sampling strata and primary
sampling units within each geographic area to
understand the survey design structure.

In [31]:
# Examine sampling design structure by state
state_design_summary = brfss.groupby("State_Name").agg(
    Respondents=("_STATE", "size"),
    Strata_Count=("_STSTR", "nunique"),
    PSU_Count=("_PSU", "nunique")
)

display(state_design_summary)

,Respondents,Strata_Count,PSU_Count
State_Name,,,
Alabama,4171,33,4171
Alaska,5537,29,5537
Arizona,5374,28,5374
Arkansas,5119,17,5119
Colorado,9437,49,9437
Connecticut,7444,37,7444
Delaware,4202,12,4202
District of Columbia,2622,4,2622
Florida,13458,231,13458


##### 3.5.5.3 Number of PSUs per Stratum

Identify sampling strata containing only one primary
sampling unit, as these may require special handling
during variance estimation.

In [32]:
# Count unique PSUs within each state and stratum
psu_per_stratum = brfss.groupby(
    ["State_Name", "_STSTR"]
)["_PSU"].nunique()

psu_per_stratum.describe()

count    2090.000000
mean      170.410526
std       306.564860
min         1.000000
25%        13.000000
50%        78.000000
75%       194.750000
max      3748.000000
Name: _PSU, dtype: float64

In [33]:
# Count strata containing only one PSU
single_psu_strata = psu_per_stratum[psu_per_stratum == 1]

print("Total strata:", len(psu_per_stratum))
print("Single-PSU strata:", len(single_psu_strata))

Total strata: 2090
Single-PSU strata: 94


##### 3.5.5.4 Interpretation of Sampling Design

**No missing values** were identified in either sampling
design variable (`_STSTR` and `_PSU`).

The dataset contains 2,090 sampling strata, of which
94 (4.50%) contain only one primary sampling unit.

PSU identifiers are unique within each geographic area
but are reused across geographic areas. Therefore,
geographic identifiers should be considered when
specifying the survey design.

Single-PSU strata may require special handling during
variance estimation but do not prevent the calculation
of weighted prevalence estimates.

All original sampling design variables are retained
for subsequent analysis.

---

## 4. Data Cleaning and Recoding

Transform the original BRFSS response codes into
analysis-ready variables for estimating weighted
state-level obesity and diabetes prevalence.

Missing and invalid responses are handled separately
for each health condition to retain as many valid
observations as possible.

### 4.1 Create a Working Copy

Create a copy of the selected BRFSS dataset to preserve
the original data during the cleaning and recoding process.

In [34]:
# Create a working copy
brfss_clean = brfss.copy()

### 4.2 Recoding Obesity Status

Recode the BMI category variable (_BMI5CAT) into
a binary obesity indicator.

- 1: Obese (BMI >= 30)
- 0: Non-obese (BMI < 30)
- NaN: Missing or invalid response

Missing values are retained rather than classified
as non-obese.

In [35]:
# Define obesity recoding rules
obesity_mapping = {
    1: 0,
    2: 0,
    3: 0,
    4: 1
}

In [36]:
# Create a binary obesity indicator
brfss_clean["Obesity"] = brfss_clean["_BMI5CAT"].map(
    obesity_mapping
)

#### 4.2.1 Validation of Obesity Recoding

Examine the distribution of the recoded obesity
indicator to verify that all original response
categories are correctly transformed.

In [37]:
# Check the distribution of the recoded variable
brfss_clean["Obesity"].value_counts(dropna=False)

Obesity
0.0    216662
1.0    106934
NaN     32562
Name: count, dtype: int64

In [38]:
# Compare original and recoded values
pd.crosstab(
    brfss_clean["_BMI5CAT"],
    brfss_clean["Obesity"],
    dropna=False
)

Obesity,0.0,1.0,NaN
_BMI5CAT,,,
1.0,5552,0,0
2.0,96272,0,0
3.0,114838,0,0
4.0,0,106934,0
NaN,0,0,32562


#### 4.2.2 Interpretation

The obesity indicator was successfully created.

- Non-obese respondents: 216,662
- Obese respondents: 106,934
- Missing responses: 32,562

All original BMI categories were correctly recoded,
and missing values were preserved.

No observations were removed during the recoding process.

### 4.3 Recoding Diabetes Status

Recode the diabetes status variable (DIABETE4)
into a binary diabetes indicator.

- 1: Diagnosed diabetes
- 0: No diabetes, pregnancy-related diabetes only,
  or prediabetes
- NaN: Unknown, refused, or missing responses

This definition excludes pregnancy-related diabetes
only and prediabetes from diagnosed diabetes prevalence.

In [39]:
# Define diabetes recoding rules
diabetes_mapping = {
    1: 1,
    2: 0,
    3: 0,
    4: 0,
    7: np.nan,
    9: np.nan
}

In [40]:
# Create a binary diabetes indicator
brfss_clean["Diabetes"] = brfss_clean["DIABETE4"].map(
    diabetes_mapping
)

#### 4.3.1 Validation of Diabetes Recoding

Verify that valid diabetes responses are correctly
transformed and that unknown, refused, and missing
responses are represented as missing values.

In [41]:
# Check the distribution of the recoded variable
brfss_clean["Diabetes"].value_counts(dropna=False)

Diabetes
0.0    303549
1.0     51827
NaN       782
Name: count, dtype: int64

In [42]:
# Compare original and recoded values
pd.crosstab(
    brfss_clean["DIABETE4"],
    brfss_clean["Diabetes"],
    dropna=False
)

Diabetes,0.0,1.0,NaN
DIABETE4,,,
1.0,0,51827,0
2.0,2671,0,0
3.0,290712,0,0
4.0,10166,0,0
7.0,0,0,580
9.0,0,0,196
NaN,0,0,6


#### 4.3.2 Interpretation

The diabetes indicator was successfully created.

- Respondents without diagnosed diabetes under the
  study definition: 303,549
- Respondents with diagnosed diabetes: 51,827
- Unknown, refused, or missing responses: 782

Pregnancy-related diabetes only and prediabetes
were classified as 0.

Unknown and refused responses were converted to
missing values.

No observations were removed during the recoding process.

### 4.4 Recoding Medicaid Status

Recode the primary health insurance variable (PRIMINS2)
into a binary primary Medicaid coverage indicator.

- 1: Medicaid reported as primary health insurance
- 0: Other reported primary coverage or no coverage
- NaN: Unknown, refused, or missing responses

This indicator is used to identify respondents who
report Medicaid as their primary health insurance,
rather than all individuals enrolled in Medicaid.

In [43]:
# Define Medicaid recoding rules
medicaid_mapping = {
    1: 0,
    2: 0,
    3: 0,
    4: 0,
    5: 1,
    6: 0,
    7: 0,
    8: 0,
    9: 0,
    10: 0,
    77: np.nan,
    88: 0,
    99: np.nan
}

In [44]:
# Create a binary primary Medicaid coverage indicator
brfss_clean["Medicaid"] = brfss_clean["PRIMINS2"].map(
    medicaid_mapping
)

#### 4.4.1 Validation of Medicaid Recoding

Verify that respondents reporting Medicaid as their
primary health insurance are correctly identified
and that unknown, refused, and missing responses
are appropriately handled.

In [45]:
# Check the distribution of the recoded variable
brfss_clean["Medicaid"].value_counts(dropna=False)

Medicaid
0.0    313964
1.0     26732
NaN     15462
Name: count, dtype: int64

In [46]:
# Compare original and recoded values
pd.crosstab(
    brfss_clean["PRIMINS2"],
    brfss_clean["Medicaid"],
    dropna=False
)

Medicaid,0.0,1.0,NaN
PRIMINS2,,,
1.0,116895,0,0
2.0,30073,0,0
3.0,117118,0,0
4.0,290,0,0
5.0,0,26732,0
6.0,118,0,0
7.0,14087,0,0
8.0,1049,0,0
9.0,8524,0,0


#### 4.4.2 Interpretation

The primary Medicaid coverage indicator was successfully
created.

- Primary Medicaid coverage: 26,732
- Other known coverage status: 313,964
- Unknown, refused, or missing responses: 15,462

Respondents reporting no health insurance coverage
were classified as 0.

Unknown and refused responses were converted to
missing values.

No observations were removed during the recoding process.

### 4.5 Validation of Cleaned Dataset

Validate the cleaned dataset to ensure that the
recoding process preserves the original observations
and produces the intended analysis variables.

#### 4.5.1 Dataset Dimensions

Compare the dimensions of the original selected dataset
and the cleaned dataset to verify that no observations
were unintentionally removed.

In [47]:
# Compare dataset dimensions
print("Original dataset:", brfss.shape)
print("Cleaned dataset:", brfss_clean.shape)

Original dataset: (356158, 8)
Cleaned dataset: (356158, 11)


#### 4.5.2 Preview of Cleaned Dataset

Inspect the cleaned dataset to verify that the original
variables and newly created binary indicators are
properly retained.

In [48]:
# Preview the cleaned dataset
display(brfss_clean.head())

,_STATE,_BMI5CAT,DIABETE4,PRIMINS2,_LLCPWT,_STSTR,_PSU,State_Name,Obesity,Diabetes,Medicaid
0,1.0,4.0,3.0,3.0,135.411848,11011.0,2.025000e+09,Alabama,1.0,0.0,0.0
1,1.0,2.0,1.0,7.0,190.376819,11011.0,2.025000e+09,Alabama,0.0,1.0,0.0
2,1.0,4.0,1.0,3.0,471.630521,11011.0,2.025000e+09,Alabama,1.0,1.0,0.0
3,1.0,3.0,1.0,3.0,473.720063,11011.0,2.025000e+09,Alabama,0.0,1.0,0.0
4,1.0,3.0,3.0,1.0,662.064923,11011.0,2.025000e+09,Alabama,0.0,0.0,0.0


#### 4.5.3 Missing Value Assessment after Recoding

Examine the number and percentage of valid and missing
responses for each newly created binary indicator.

In [49]:
# Select recoded variables
recoded_columns = ["Obesity", "Diabetes", "Medicaid"]

In [50]:
# Summarize valid and missing responses
recoding_summary = pd.DataFrame({
    "Valid_Count": brfss_clean[recoded_columns].notna().sum(),
    "Missing_Count": brfss_clean[recoded_columns].isna().sum(),
    "Missing_Pct": (
        brfss_clean[recoded_columns].isna().mean() * 100
    )
})

display(recoding_summary.round(2))

,Valid_Count,Missing_Count,Missing_Pct
Obesity,323596,32562,9.14
Diabetes,355376,782,0.22
Medicaid,340696,15462,4.34


#### 4.5.4 Validation of Original Variables

Verify that all original selected variables remain
unchanged after creating the binary indicators.

In [51]:
# Verify that original variables are unchanged
original_variables_preserved = brfss_clean[
    brfss.columns
].equals(brfss)

print("Original variables preserved:", original_variables_preserved)

Original variables preserved: True


In [52]:
# Verify that recoded variables contain only valid binary values
for col in recoded_columns:
    is_valid = brfss_clean[col].dropna().isin([0, 1]).all()
    print(f"{col}: {is_valid}")

Obesity: True
Diabetes: True
Medicaid: True


### 4.6 Summary of Data Cleaning

The data cleaning and recoding process was successfully
completed.

- All 356,158 original observations were retained.
- Three binary indicators (Obesity, Diabetes, and
  Medicaid) were created.
- Missing, unknown, and refused responses were
  appropriately handled.
- All original variables and survey weights were preserved.
- All recoded variables contain only 0, 1, or NaN.

The cleaned dataset contains 356,158 observations
and 11 variables.

Missing responses will be excluded separately for each
health condition during prevalence estimation to maximize
the use of available data.

---

## 5. Medicaid Subpopulation Analysis

Assess the feasibility of estimating state-level obesity
and diabetes prevalence among respondents who report
Medicaid as their primary health insurance.

Examine valid sample sizes and missing response patterns
for both the overall adult sample and the primary
Medicaid subpopulation before weighted prevalence
estimation.

### 5.1 Create Medicaid Subpopulation

Create a subset of respondents who report Medicaid
as their primary health insurance (**Medicaid = 1**).

The full cleaned dataset is retained for overall adult
prevalence estimation and subsequent survey design-based
analysis.

In [53]:
# Create Medicaid subpopulation
brfss_medicaid = brfss_clean.loc[
    brfss_clean["Medicaid"] == 1
].copy()

In [54]:
# Examine Medicaid subpopulation dimensions
brfss_medicaid.shape

(26732, 11)

### 5.2 Valid Sample Size Comparison

Compare the number of valid obesity and diabetes
responses between the overall adult sample and the
primary Medicaid subpopulation.

Valid sample sizes are calculated separately for each
health condition.

In [55]:
# Compare total and valid sample sizes
population_comparison = pd.DataFrame({
    "Overall": {
        "Total_Respondents": len(brfss_clean),
        "Valid_Obesity": brfss_clean["Obesity"].notna().sum(),
        "Valid_Diabetes": brfss_clean["Diabetes"].notna().sum()
    },
    "Medicaid": {
        "Total_Respondents": len(brfss_medicaid),
        "Valid_Obesity": brfss_medicaid["Obesity"].notna().sum(),
        "Valid_Diabetes": brfss_medicaid["Diabetes"].notna().sum()
    }
})

display(population_comparison)

,Overall,Medicaid
Total_Respondents,356158,26732
Valid_Obesity,323596,23991
Valid_Diabetes,355376,26651


### 5.3 Missing Response Rate Comparison

Compare missing response rates for obesity and diabetes
between the overall adult sample and the primary
Medicaid subpopulation.

Differences in missing response rates may affect
the reliability and comparability of prevalence estimates.

In [56]:
# Compare missing response percentages
missing_comparison = pd.DataFrame({
    "Overall": (
        brfss_clean[["Obesity", "Diabetes"]].isna().mean() * 100
    ),
    "Medicaid": (
        brfss_medicaid[["Obesity", "Diabetes"]].isna().mean() * 100
    )
})

display(missing_comparison.round(2))

,Overall,Medicaid
Obesity,9.14,10.25
Diabetes,0.22,0.30


#### 5.3.1 Visualization of Missing Response Rates

Visualize differences in missing response rates
between the two population groups.

In [57]:
# Reshape data for visualization
missing_comparison_long = (
    missing_comparison
    .rename_axis("Variable")
    .reset_index()
    .melt(
        id_vars="Variable",
        var_name="Population",
        value_name="Missing_Pct"
    )
)

In [58]:
# Visualize missing response rates
alt.Chart(missing_comparison_long).mark_bar().encode(
    x=alt.X("Variable:N", title="Health Condition"),
    xOffset="Population:N",
    y=alt.Y("Missing_Pct:Q", title="Missing Responses (%)"),
    color=alt.Color("Population:N", title="Population"),
    tooltip=[
        "Variable:N",
        "Population:N",
        alt.Tooltip("Missing_Pct:Q", format=".2f")
    ]
).properties(
    title="Missing Response Rates by Population",
    width=500,
    height=350
)

alt.Chart(...)

#### 5.3.2 Interpretation of Population Comparison

A total of 26,732 respondents reported Medicaid as
their primary health insurance, representing
approximately **7.51%** of the overall sample.

Among these respondents, 23,991 have valid obesity
responses and 26,651 have valid diabetes responses.

Missing response rates are slightly higher in the
Medicaid subpopulation than in the overall sample:

- **Obesity:** 10.25% compared to 9.14%
- **Diabetes:** 0.30% compared to 0.22%

Although the aggregate Medicaid sample is relatively
large, state-level valid sample sizes must be examined
before determining the feasibility of Medicaid-specific
prevalence estimation.

---

### 5.4 Valid Sample Sizes by State

Examine state-level valid sample sizes for obesity
and diabetes in both the overall adult sample and
the primary Medicaid subpopulation.

#### 5.4.1 Overall Valid Sample Sizes by State

Calculate the total number of respondents and valid
health condition responses within each geographic area
for the overall adult sample.

In [59]:
# Calculate overall valid sample sizes by state
overall_valid_by_state = brfss_clean.groupby(
    "State_Name"
).agg(
    Total_Overall=("State_Name", "size"),
    Valid_Obesity_Overall=("Obesity", "count"),
    Valid_Diabetes_Overall=("Diabetes", "count")
).reset_index()

display(overall_valid_by_state)

,State_Name,Total_Overall,Valid_Obesity_Overall,Valid_Diabetes_Overall
0,Alabama,4171,3905,4165
1,Alaska,5537,5121,5528
2,Arizona,5374,4855,5367
3,Arkansas,5119,4676,5106
4,Colorado,9437,8638,9416
5,Connecticut,7444,6465,7424
6,Delaware,4202,3814,4197
7,District of Columbia,2622,2418,2617
8,Florida,13458,12098,13417
9,Georgia,6191,5542,6168


#### 5.4.2 Medicaid Valid Sample Sizes by State

Calculate the total number of Medicaid respondents
and valid health condition responses within each
geographic area.

In [60]:
# Calculate Medicaid valid sample sizes by state
medicaid_valid_by_state = brfss_medicaid.groupby(
    "State_Name"
).agg(
    Total_Medicaid=("State_Name", "size"),
    Valid_Obesity_Medicaid=("Obesity", "count"),
    Valid_Diabetes_Medicaid=("Diabetes", "count")
).reset_index()

display(medicaid_valid_by_state)

,State_Name,Total_Medicaid,Valid_Obesity_Medicaid,Valid_Diabetes_Medicaid
0,Alabama,247,226,247
1,Alaska,529,477,528
2,Arizona,184,165,184
3,Arkansas,414,374,412
4,Colorado,861,746,857
5,Connecticut,517,433,516
6,Delaware,284,252,284
7,District of Columbia,307,267,306
8,Florida,608,525,607
9,Georgia,315,275,312


#### 5.4.3 Merge State-Level Valid Sample Sizes

Combine the overall and Medicaid sample size summaries
to facilitate state-level comparisons.

In [61]:
# Merge overall and Medicaid sample size summaries
state_valid_comparison = overall_valid_by_state.merge(
    medicaid_valid_by_state,
    on="State_Name",
    how="left",
    validate="one_to_one"
)

display(state_valid_comparison)

,State_Name,Total_Overall,Valid_Obesity_Overall,Valid_Diabetes_Overall,Total_Medicaid,Valid_Obesity_Medicaid,Valid_Diabetes_Medicaid
0,Alabama,4171,3905,4165,247,226,247
1,Alaska,5537,5121,5528,529,477,528
2,Arizona,5374,4855,5367,184,165,184
3,Arkansas,5119,4676,5106,414,374,412
4,Colorado,9437,8638,9416,861,746,857
5,Connecticut,7444,6465,7424,517,433,516
6,Delaware,4202,3814,4197,284,252,284
7,District of Columbia,2622,2418,2617,307,267,306
8,Florida,13458,12098,13417,608,525,607
9,Georgia,6191,5542,6168,315,275,312


#### 5.4.4 Descriptive Statistics of Valid Sample Sizes

Examine the distribution of state-level valid sample
sizes to identify potential limitations associated
with small Medicaid samples.

In [62]:
# Examine descriptive statistics of valid sample sizes
state_valid_comparison[[
    "Valid_Obesity_Overall",
    "Valid_Diabetes_Overall",
    "Valid_Obesity_Medicaid",
    "Valid_Diabetes_Medicaid"
]].describe().round(2)

,Valid_Obesity_Overall,Valid_Diabetes_Overall,Valid_Obesity_Medicaid,Valid_Diabetes_Medicaid
count,50.00,50.00,50.00,50.00
mean,6471.92,7107.52,479.82,533.02
std,3299.30,3683.77,345.27,390.40
min,1809.00,1917.00,108.00,115.00
25%,4030.50,4289.25,256.50,292.00
50%,5494.00,5956.50,409.50,454.00
75%,8293.75,9308.50,569.25,627.25
max,17382.00,19459.00,2083.00,2422.00


#### 5.4.5 Identify States with Small Medicaid Samples

Identify geographic areas with the smallest valid
Medicaid sample sizes for obesity and diabetes.

Small sample sizes may indicate potential limitations
in the precision of prevalence estimates.

In [63]:
# Identify the 10 smallest Medicaid obesity samples
state_valid_comparison[[
    "State_Name",
    "Total_Medicaid",
    "Valid_Obesity_Medicaid"
]].nsmallest(10, "Valid_Obesity_Medicaid")

,State_Name,Total_Medicaid,Valid_Obesity_Medicaid
49,Wyoming,116,108
27,New Hampshire,173,159
2,Arizona,184,165
10,Guam,186,169
36,Pennsylvania,192,177
40,South Dakota,215,199
39,South Carolina,231,206
41,Tennessee,230,210
0,Alabama,247,226
32,North Dakota,279,246


In [64]:
# Identify the 10 smallest Medicaid diabetes samples
state_valid_comparison[[
    "State_Name",
    "Total_Medicaid",
    "Valid_Diabetes_Medicaid"
]].nsmallest(10, "Valid_Diabetes_Medicaid")

,State_Name,Total_Medicaid,Valid_Diabetes_Medicaid
49,Wyoming,116,115
27,New Hampshire,173,173
10,Guam,186,182
2,Arizona,184,184
36,Pennsylvania,192,191
40,South Dakota,215,215
41,Tennessee,230,228
39,South Carolina,231,231
0,Alabama,247,247
31,North Carolina,276,276


### 5.5 Summary of Subpopulation Analysis

All 50 observed geographic areas have valid responses for both outcomes in both populations. Medicaid valid sample sizes range from 108 to 2,083 for obesity and from 115 to 2,422 for diabetes. No area is excluded solely on the basis of sample size.

## 6. Weighted Prevalence Estimation

Calculate state-level prevalence using the final BRFSS survey weight (`_LLCPWT`):

$$P = 100 \times \frac{\sum_i w_i y_i}{\sum_i w_i}$$

Use separate valid-response denominators for obesity and diabetes. Estimate prevalence for all adults and for adults reporting Medicaid as their primary insurance. Also calculate valid sample size and Kish effective sample size, $(\sum w_i)^2 / \sum w_i^2$, as a weight-based precision diagnostic, not a full survey-design variance estimate.

In [65]:
def weighted_prevalence(data):
    rows = []
    for (state_code, state_name), group in data.groupby(["_STATE", "State_Name"]):
        row = {"State_FIPS": f"{int(state_code):02d}",
               "State": state_name, "Year": 2025}
        for outcome in ["Obesity", "Diabetes"]:
            valid = group.loc[group[outcome].notna() &
                              np.isfinite(group["_LLCPWT"]) &
                              (group["_LLCPWT"] > 0)]
            w = valid["_LLCPWT"]
            row[f"{outcome}_N"] = len(valid)
            row[f"{outcome}_Pct"] = (
                100 * np.average(valid[outcome], weights=w) if len(valid) else np.nan
            )
            row[f"{outcome}_Effective_N"] = (
                w.sum() ** 2 / (w ** 2).sum() if len(valid) else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows)

### 6.1 Overall Adult Prevalence

In [66]:
overall_prevalence = weighted_prevalence(brfss_clean)
display(overall_prevalence.round(2))

,State_FIPS,State,Year,Obesity_N,Obesity_Pct,Obesity_Effective_N,Diabetes_N,Diabetes_Pct,Diabetes_Effective_N
0,01,Alabama,2025,3905,39.71,2330.68,4165,15.63,2488.41
1,02,Alaska,2025,5121,33.03,2316.57,5528,8.25,2520.65
2,04,Arizona,2025,4855,32.27,1848.31,5367,13.05,2060.65
3,05,Arkansas,2025,4676,36.73,2635.83,5106,14.25,2848.98
4,08,Colorado,2025,8638,25.66,6323.59,9416,8.76,6862.59
5,09,Connecticut,2025,6465,30.38,3636.89,7424,10.94,4223.31
6,10,Delaware,2025,3814,36.31,1829.09,4197,13.42,2029.31
7,11,District of Columbia,2025,2418,24.16,1327.99,2617,9.07,1426.17
8,12,Florida,2025,12098,29.98,3106.56,13417,13.25,3460.99
9,13,Georgia,2025,5542,35.92,2794.49,6168,13.03,3113.51


### 6.2 Primary Medicaid Prevalence

In [67]:
medicaid_prevalence = weighted_prevalence(brfss_medicaid)
display(medicaid_prevalence.round(2))

,State_FIPS,State,Year,Obesity_N,Obesity_Pct,Obesity_Effective_N,Diabetes_N,Diabetes_Pct,Diabetes_Effective_N
0,01,Alabama,2025,226,52.42,144.91,247,15.88,159.15
1,02,Alaska,2025,477,29.39,213.47,528,10.25,241.31
2,04,Arizona,2025,165,34.93,77.02,184,20.40,86.76
3,05,Arkansas,2025,374,47.63,211.34,412,20.80,234.95
4,08,Colorado,2025,746,30.87,547.77,857,9.54,619.98
5,09,Connecticut,2025,433,36.33,230.13,516,16.32,283.31
6,10,Delaware,2025,252,43.40,131.49,284,12.70,152.80
7,11,District of Columbia,2025,267,36.80,159.56,306,13.68,184.70
8,12,Florida,2025,525,32.17,152.81,607,13.72,181.52
9,13,Georgia,2025,275,42.56,135.12,312,12.32,158.43


## 7. Comparison and Reliability Assessment

Compare the two estimates and their sample sizes. Effective sample size reflects unequal weighting only; it does not account for stratification or clustering. No formal confidence intervals or suppression decisions are made here.

In [68]:
prevalence_comparison = overall_prevalence.merge(
    medicaid_prevalence, on=["State_FIPS", "State", "Year"],
    suffixes=("_Overall", "_Medicaid"), validate="one_to_one"
)
for outcome in ["Obesity", "Diabetes"]:
    prevalence_comparison[f"{outcome}_Difference_Pp"] = (
        prevalence_comparison[f"{outcome}_Pct_Medicaid"] -
        prevalence_comparison[f"{outcome}_Pct_Overall"]
    )
display(prevalence_comparison.round(2))
display(prevalence_comparison.filter(regex="Effective_N|Difference_Pp").describe().round(2))

,State_FIPS,State,Year,Obesity_N_Overall,Obesity_Pct_Overall,Obesity_Effective_N_Overall,Diabetes_N_Overall,Diabetes_Pct_Overall,Diabetes_Effective_N_Overall,Obesity_N_Medicaid,Obesity_Pct_Medicaid,Obesity_Effective_N_Medicaid,Diabetes_N_Medicaid,Diabetes_Pct_Medicaid,Diabetes_Effective_N_Medicaid,Obesity_Difference_Pp,Diabetes_Difference_Pp
0,01,Alabama,2025,3905,39.71,2330.68,4165,15.63,2488.41,226,52.42,144.91,247,15.88,159.15,12.71,0.25
1,02,Alaska,2025,5121,33.03,2316.57,5528,8.25,2520.65,477,29.39,213.47,528,10.25,241.31,-3.64,1.99
2,04,Arizona,2025,4855,32.27,1848.31,5367,13.05,2060.65,165,34.93,77.02,184,20.40,86.76,2.66,7.35
3,05,Arkansas,2025,4676,36.73,2635.83,5106,14.25,2848.98,374,47.63,211.34,412,20.80,234.95,10.91,6.55
4,08,Colorado,2025,8638,25.66,6323.59,9416,8.76,6862.59,746,30.87,547.77,857,9.54,619.98,5.21,0.78
5,09,Connecticut,2025,6465,30.38,3636.89,7424,10.94,4223.31,433,36.33,230.13,516,16.32,283.31,5.94,5.39
6,10,Delaware,2025,3814,36.31,1829.09,4197,13.42,2029.31,252,43.40,131.49,284,12.70,152.80,7.09,-0.73
7,11,District of Columbia,2025,2418,24.16,1327.99,2617,9.07,1426.17,267,36.80,159.56,306,13.68,184.70,12.64,4.61
8,12,Florida,2025,12098,29.98,3106.56,13417,13.25,3460.99,525,32.17,152.81,607,13.72,181.52,2.19,0.47
9,13,Georgia,2025,5542,35.92,2794.49,6168,13.03,3113.51,275,42.56,135.12,312,12.32,158.43,6.65,-0.71


,Obesity_Effective_N_Overall,Diabetes_Effective_N_Overall,Obesity_Effective_N_Medicaid,Diabetes_Effective_N_Medicaid,Obesity_Difference_Pp,Diabetes_Difference_Pp
count,50.00,50.00,50.00,50.00,50.00,50.00
mean,3438.55,3771.00,265.24,296.38,6.90,1.74
std,1979.57,2193.97,208.64,238.49,3.42,2.72
min,140.75,156.84,32.37,35.75,-3.64,-3.74
25%,2173.47,2409.07,145.52,159.13,4.90,-0.10
50%,2914.98,3173.12,212.41,243.36,6.87,1.17
75%,4714.79,5171.35,330.74,361.06,9.34,3.46
max,10539.03,11777.12,1285.97,1497.47,13.78,10.74


For the primary regression, the overall adult estimates are a conservative starting point given their larger observed sample sizes. Medicaid-primary estimates are a population-aligned alternative for sensitivity analysis, but do not represent all Medicaid enrollees. Both BRFSS populations cover adults, whereas the Medicaid prescription and enrollment data may include children. The final feature choice should be agreed upon by the team after reviewing the estimates and diagnostics. Do not combine the two alternative sets of prevalence features by default.

## 8. Final Data Export

Export one row per observed geographic area and year for each population. Retain state names and two-digit FIPS identifiers for merging, prevalence in percentage units (0–100), and sample-size diagnostics. The comparison file is for review, not a separate regression input. Preserve DC and territories in the exports; the team should align geographic coverage with the other datasets during merging. California, Mississippi and Nevada are absent from the observed BRFSS data and are not imputed. Annual prevalence values can be joined to quarterly data using state and year.

In [69]:
for result in [overall_prevalence, medicaid_prevalence]:
    assert len(result) == brfss_clean["_STATE"].nunique()
    assert not result.duplicated(["State_FIPS", "Year"]).any()
    for outcome in ["Obesity", "Diabetes"]:
        assert result[f"{outcome}_Pct"].between(0, 100).all()
        assert (result[f"{outcome}_Effective_N"] <= result[f"{outcome}_N"] + 1e-8).all()

# Save full-precision estimates and diagnostics
outputs = {
    "brfss_2025_overall.csv": overall_prevalence,
    "brfss_2025_medicaid.csv": medicaid_prevalence,
    "brfss_2025_comparison.csv": prevalence_comparison
}
for filename, result in outputs.items():
    result.to_csv(filename, index=False)
    print(f"Saved {filename}: {result.shape}")

Saved brfss_2025_overall.csv: (50, 9)
Saved brfss_2025_medicaid.csv: (50, 9)
Saved brfss_2025_comparison.csv: (50, 17)


When importing the CSV files, use `pd.read_csv(filename, dtype={"State_FIPS": str})` to preserve leading zeros. Merge the selected prevalence dataset with the quarterly Medicaid data on standardized state identifier and year, using a many-to-one merge. Sample-size diagnostics are not intended as regression predictors.